# **NOTEBOOK 1 - Ingesta, exploración y verificación de integridad.**

# **CARGAR CSV EN SPARK**

In [0]:
#Cargamos el dataset en un DataFrame de Spark

df = spark.read.csv(
    "/Volumes/workspace/proyecto_bigdata/credit_card_fraud/creditcard.csv", 
    header=True, #Primera fila contiene los nombres de las columnas
    inferSchema=True # Spark infiere los tipos de datos
)
#----------------------------------------------------------------------
#Despues, hacemos una confirmacion de cuantas filas y columnas tiene el Dataset

filas = df.count()
columnas = len(df.columns)

print(f"El dataset contiene {filas} filas, {columnas} columnas y el nombre de las columnas {df.columns}")



# **REVISAR TIPOS DE DATOS Y VALORES NULOS**

In [0]:
#Mostramos el esquema del dataset (Tipos de datos)
print("El esquema del dataset es:")
df.printSchema()

#Contamos valores nulos por columna
#----------------------------------------------------------------------
from pyspark.sql.functions import col, sum as spark_sum

valores_nulos = df.select([spark_sum(col(columna).isNull().cast("int")).alias(columna) for columna in df.columns])

valores_nulos.show()
#Resultado correcto: todo en 0


# **ANALIZAR LA DISTRIBUCION DE CLASES**

In [0]:
#IMPORTAMOS LIBRERIAS NECESARIAS
#"COUNT" sirve para contar filas y "ROUND" para redondear numeros

from pyspark.sql.functions import count, round as spark_round

print("===== DISTRIBUCION DE CLASES=====")
distribucion = df.groupBy("Class").agg(count("*").alias("cantidad")).orderBy("Class")

distribucion.show()


#Calculamos el porcentaje de transacciones fraudulentas
total = df.count() #CUENTE TODAS LAS FILAS DEL DATASET

fraudes = df.filter(col("Class") == 1).count() #FILTRA SOLO LAS FILAS DONDE CLASS == 1(fraude) Y CUENTE LAS FILAS

legitimas = total - fraudes

print(f"\nTotal de transacciones: {total:,}")
print(f"Transacciones legítimas (Class=0): {legitimas:,} ({legitimas/total*100:.3f}%)")
print(f"Transacciones fraudulentas (Class=1): {fraudes:,} ({fraudes/total*100:.3f}%)")
print(f"\nRatio de desbalance: 1 fraude por cada {legitimas//fraudes} transacciones legítimas")

# ----> Los f"..." son f-strings, una forma de meter variables dentro de texto. Lo que está entre {} se reemplaza por el valor real. Dos detalles de formato que vale la pena entender:
# ----> :, agrega separadores de miles para que 284807 se muestre como 284,807, más legible.
# ----> :.3f muestra el número con exactamente 3 decimales, por eso el porcentaje aparece como 0.172% en lugar de algo como 0.17232819...%.
# ----> El // en el último print es división entera, o sea que descarta los decimales. 284315 // 492 da 578, lo que significa "por cada fraude hay 578 transacciones legítimas".


# **Estadisticas descriptivas de las columnas clave**

In [0]:
# Estadísticas descriptivas de Time y Amount
# Estas son las únicas columnas no transformadas por PCA
# SE EXTRAEN LOS DATOS DE AMOUNT Y TIME DEL DATASET
print("=== ESTADÍSTICAS DE TIME Y AMOUNT ===")
df.select("Time", "Amount").describe().show()  # ACCIÓN (show es la acción) calcula 5 estadisticas automaticamente: cuantos valores hay, promedio, desviacion estandar, minimo y maximo
 
# Revisamos también el rango de Amount para detectar valores extremos
from pyspark.sql.functions import min as spark_min, max as spark_max, avg

print("=== RANGO DE MONTOS POR CLASE ===")
df.groupBy("Class").agg(
    spark_min("Amount").alias("monto_minimo"),
    spark_max("Amount").alias("monto_maximo"),
    avg("Amount").alias("monto_promedio")
).orderBy("Class").show()

# **Notebook 2 — ETL y Preprocesamiento**

## **Registrar el DataFrame como tabla tempora**l

In [0]:
# Registramos el DataFrame como una tabla temporal
# Esto nos permite consultarlo usando SQL directamente
df.createOrReplaceTempView("transacciones")

print("Tabla temporal 'transacciones' creada correctamente.")

# **Comparar el comportamiento de fraudes vs legítimas**

In [0]:
# Comparamos las principales estadísticas entre transacciones 
# fraudulentas y legítimas
spark.sql("""
    SELECT 
        Class,
        COUNT(*) as total_transacciones,
        ROUND(AVG(Amount), 2) as monto_promedio,
        ROUND(MAX(Amount), 2) as monto_maximo,
        ROUND(MIN(Amount), 2) as monto_minimo,
        ROUND(AVG(Time), 2) as tiempo_promedio
    FROM transacciones
    GROUP BY Class
    ORDER BY Class
""").show()

# **Identificar transacciones de alto monto**

In [0]:
# Identificamos cuántas transacciones superan ciertos umbrales de monto
# Esto nos ayuda a entender la distribución antes de normalizar
spark.sql("""
    SELECT 
        Class,
        COUNT(*) as total,
        SUM(CASE WHEN Amount > 1000 THEN 1 ELSE 0 END) as sobre_1000,
        SUM(CASE WHEN Amount > 5000 THEN 1 ELSE 0 END) as sobre_5000,
        SUM(CASE WHEN Amount > 10000 THEN 1 ELSE 0 END) as sobre_10000
    FROM transacciones
    GROUP BY Class
    ORDER BY Class
""").show()

# **Calcular media y desviación estándar**

In [0]:
from pyspark.sql.functions import col, avg, stddev

# Calculamos los valores que necesitamos para la fórmula de normalización
# avg = promedio, stddev = desviación estándar
stats = df.select(
    avg("Amount").alias("media_amount"),
    stddev("Amount").alias("std_amount"),
    avg("Time").alias("media_time"),
    stddev("Time").alias("std_time")
).collect()[0]

# Mostramos los valores calculados
print("=== VALORES PARA NORMALIZACIÓN ===")
print(f"Amount  — Media: {stats['media_amount']:.4f} | Desv. estándar: {stats['std_amount']:.4f}")
print(f"Time    — Media: {stats['media_time']:.4f} | Desv. estándar: {stats['std_time']:.4f}")

# **Aplicar la normalización**

In [0]:
# Aplicamos la fórmula de normalización estándar directamente
# Fórmula: (valor - media) / desviación_estándar
# Resultado: cada columna queda con media 0 y desviación estándar 1

df_trabajo = df.withColumn(
    "Amount_scaled",
    (col("Amount") - stats["media_amount"]) / stats["std_amount"]
).withColumn(
    "Time_scaled",
    (col("Time") - stats["media_time"]) / stats["std_time"]
)

print("Normalización aplicada correctamente.")
print(f"Columnas totales ahora: {len(df_trabajo.columns)}")

# **Verificar que la normalización funcionó**

In [0]:
# Verificamos comparando los valores originales con los normalizados
print("=== COMPARACIÓN ANTES Y DESPUÉS ===")
df_trabajo.select(
    "Amount", 
    "Amount_scaled",
    "Time", 
    "Time_scaled"
).show(10)

# Confirmamos que Amount_scaled tiene media ≈ 0 y desviación estándar ≈ 1
print("=== ESTADÍSTICAS DE LAS COLUMNAS NORMALIZADAS ===")
df_trabajo.select(
    avg("Amount_scaled").alias("media_amount_scaled"),
    stddev("Amount_scaled").alias("std_amount_scaled"),
    avg("Time_scaled").alias("media_time_scaled"),
    stddev("Time_scaled").alias("std_time_scaled")
).show()

# **Seleccionar las columnas finales**

In [0]:
# Definimos qué columnas usará el modelo
# Usamos Amount_scaled y Time_scaled (normalizadas) en lugar de las originales
# Las columnas V1-V28 se mantienen igual porque ya estaban en escala correcta

columnas_features = ["Amount_scaled", "Time_scaled"] + \
                    [f"V{i}" for i in range(1, 29)]

# Creamos el dataset final solo con esas columnas más la variable objetivo
df_final = df_trabajo.select(columnas_features + ["Class"])

# Verificamos
print(f"Columnas seleccionadas: {len(df_final.columns)}")
print(f"Features: {len(columnas_features)}")
print(f"Variable objetivo: Class")
df_final.show(5)

# **Calcular el peso para manejar el desbalance**

In [0]:
# Calculamos el peso que le daremos a la clase minoritaria (fraudes)
# Esto le indica al modelo que los errores en fraudes deben penalizarse más

total = df_final.count()
fraudes = df_final.filter(col("Class") == 1).count()
legitimas = total - fraudes

# Fórmula estándar para peso de clase minoritaria
peso_fraude = legitimas / fraudes

print("=== ESTRATEGIA PARA MANEJO DEL DESBALANCE ===")
print(f"Total transacciones: {total:,}")
print(f"Legítimas (Class=0): {legitimas:,}")
print(f"Fraudes (Class=1): {fraudes:,}")
print(f"\nPeso asignado a fraudes: {peso_fraude:.2f}")
print(f"Interpretación: cada fraude cuenta como {peso_fraude:.0f} transacciones legítimas")

# **Dividir en entrenamiento y prueba**

In [0]:
from pyspark.sql.functions import when, lit

# Agregamos el peso calculado como columna
# Class=1 (fraude) recibe el peso alto, Class=0 recibe peso 1.0
df_final = df_final.withColumn(
    "peso_clase",
    when(col("Class") == 1, peso_fraude).otherwise(1.0)
)

# Dividimos: 80% entrenamiento, 20% prueba
# El seed=42 garantiza que la división sea siempre la misma si se repite
df_entrenamiento, df_prueba = df_final.randomSplit([0.8, 0.2], seed=42)

# Verificamos la división
print("=== DIVISIÓN DEL DATASET ===")
print(f"Total original:      {df_final.count():,}")
print(f"Entrenamiento (80%): {df_entrenamiento.count():,}")
print(f"Prueba (20%):        {df_prueba.count():,}")

# Verificamos que el desbalance se mantiene proporcionalmente en ambas partes
print("\n=== DISTRIBUCIÓN DE CLASES EN CADA CONJUNTO ===")
print("Entrenamiento:")
df_entrenamiento.groupBy("Class").count().orderBy("Class").show()
print("Prueba:")
df_prueba.groupBy("Class").count().orderBy("Class").show()

# **Notebook 3 — Modelo base y pruebas preliminares**

# **Confirmar los datos disponibles**

In [0]:
# Confirmamos que los DataFrames del Notebook 2 siguen disponibles
print("=== VERIFICACIÓN DE DATOS ===")
print(f"Entrenamiento: {df_entrenamiento.count():,} filas")
print(f"Prueba:        {df_prueba.count():,} filas")
print(f"Columnas:      {df_entrenamiento.columns}")

# **Preparar los datos para Spark MLlib**

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Definimos las columnas que usará el modelo como features
# Son las mismas 30 columnas que seleccionamos en el Notebook 2
columnas_features = ["Amount_scaled", "Time_scaled"] + \
                    [f"V{i}" for i in range(1, 29)]

# VectorAssembler combina todas las features en un solo vector
ensamblador = VectorAssembler(
    inputCols=columnas_features,
    outputCol="features"
)

# Aplicamos el ensamblador a entrenamiento y prueba
df_train = ensamblador.transform(df_entrenamiento)
df_test = ensamblador.transform(df_prueba)

# Verificamos que la columna features fue creada
print("=== DATASET LISTO PARA EL MODELO ===")
print(f"Columnas en df_train: {df_train.columns}")
df_train.select("features", "Class", "peso_clase").show(3, truncate=False)

# **Entrenar el modelo**

In [0]:
# Configuramos el modelo de Regresión Logística
# weightCol le indica al modelo que use la columna peso_clase
# que calculamos en el Notebook 2
modelo_lr = LogisticRegression(
    featuresCol="features",      # Columna con el vector de features
    labelCol="Class",            # Columna que queremos predecir
    weightCol="peso_clase",      # Peso para manejar el desbalance
    maxIter=10                   # Número máximo de iteraciones
)

# Entrenamos el modelo con los datos de entrenamiento
print("Entrenando el modelo...")
modelo_entrenado = modelo_lr.fit(df_train)
print("Modelo entrenado correctamente.")

# Aplicamos el modelo sobre los datos de prueba
predicciones = modelo_entrenado.transform(df_test)

# Revisamos las primeras predicciones
print("\n=== PRIMERAS PREDICCIONES ===")
predicciones.select(
    "Class",
    "prediction",
    "probability"
).show(10, truncate=False)

# **Evaluar el modelo**

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col, sum as spark_sum, when

# ── Métrica 1: AUPRC ──────────────────────────────────────────────
evaluador_auprc = BinaryClassificationEvaluator(
    labelCol="Class",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)
auprc = evaluador_auprc.evaluate(predicciones)

# ── Métrica 2: AUC-ROC ────────────────────────────────────────────
evaluador_auc = BinaryClassificationEvaluator(
    labelCol="Class",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = evaluador_auc.evaluate(predicciones)

# ── Matriz de confusión con DataFrame ────────────────────────────
# Calculamos cada celda directamente con filtros
vn = predicciones.filter((col("Class") == 0) & (col("prediction") == 0)).count()
fp = predicciones.filter((col("Class") == 0) & (col("prediction") == 1)).count()
fn = predicciones.filter((col("Class") == 1) & (col("prediction") == 0)).count()
vp = predicciones.filter((col("Class") == 1) & (col("prediction") == 1)).count()

# ── Calculamos métricas adicionales manualmente ──────────────────
total = vn + fp + fn + vp
precision_global = (vn + vp) / total
precision_fraude = vp / (vp + fp) if (vp + fp) > 0 else 0
recall_fraude    = vp / (vp + fn) if (vp + fn) > 0 else 0
f1_fraude        = (2 * precision_fraude * recall_fraude) / \
                   (precision_fraude + recall_fraude) \
                   if (precision_fraude + recall_fraude) > 0 else 0

# ── Resultados ────────────────────────────────────────────────────
print("=== MATRIZ DE CONFUSIÓN ===")
print(f"Verdaderos Negativos  (legítima → legítima): {vn:,}")
print(f"Falsos Positivos      (legítima → fraude):   {fp:,}")
print(f"Falsos Negativos      (fraude → legítima):   {fn:,}")
print(f"Verdaderos Positivos  (fraude → fraude):     {vp:,}")

print("\n=== RESUMEN DE MÉTRICAS ===")
print(f"AUPRC:              {auprc:.4f}")
print(f"AUC-ROC:            {auc:.4f}")
print(f"Precisión global:   {precision_global:.4f}")
print(f"Precisión fraude:   {precision_fraude:.4f}")
print(f"Recall fraude:      {recall_fraude:.4f}")
print(f"F1-Score fraude:    {f1_fraude:.4f}")

print("\n=== INTERPRETACIÓN ===")
print(f"De {vp + fn} fraudes reales en el conjunto de prueba:")
print(f"  → Detectados correctamente: {vp} ({vp/(vp+fn)*100:.1f}%)")
print(f"  → No detectados (escaparon): {fn} ({fn/(vp+fn)*100:.1f}%)")